# Day 3 — Multimodal Indexing

Build one FAISS index for text chunks, image captions, and image embeddings. Keep every vector linked to its source metadata.

## 1. Install dependencies

Install FAISS and the libraries used to load CLIP and the Day 2 outputs.

In [1]:
!pip -q install faiss-cpu transformers torch pillow pandas

## 2. Imports

Import the indexing, embedding, metadata, and file-handling libraries.

In [2]:
from pathlib import Path
import json
import torch
import torch.nn.functional as F
import faiss
import pandas as pd
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from google.colab import files

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


## 3. Upload Day 2 outputs

Upload image_captions_metadata.json, text_chunks_metadata.json, multimodal_embeddings.pt, and the extracted image files from the Day 2 image folder.

In [3]:
uploaded = files.upload()

json_files = [name for name in uploaded if name.endswith(".json")]
pt_files = [name for name in uploaded if name.endswith(".pt")]
image_extensions = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
uploaded_images = [name for name in uploaded if Path(name).suffix.lower() in image_extensions]

image_metadata_file = next((n for n in json_files if "image_captions_metadata" in n), None)
text_metadata_file = next((n for n in json_files if "text_chunks_metadata" in n), None)
embeddings_file = pt_files[0] if pt_files else None

if not image_metadata_file or not text_metadata_file or not embeddings_file:
    raise ValueError("Upload both metadata JSON files and multimodal_embeddings.pt.")

print("Image metadata:", image_metadata_file)
print("Text metadata:", text_metadata_file)
print("Embeddings:", embeddings_file)
print("Image files:", len(uploaded_images))

Saving image_captions_metadata (1).json to image_captions_metadata (1).json
Saving multimodal_embeddings (1).pt to multimodal_embeddings (1).pt
Saving text_chunks_metadata (1).json to text_chunks_metadata (1).json
Image metadata: image_captions_metadata (1).json
Text metadata: text_chunks_metadata (1).json
Embeddings: multimodal_embeddings (1).pt
Image files: 0


## 4. Load metadata and embeddings

Load the Day 2 records and vector tensors.

In [4]:
with open(image_metadata_file, encoding="utf-8") as f:
    image_records = json.load(f)
with open(text_metadata_file, encoding="utf-8") as f:
    text_records = json.load(f)

saved = torch.load(embeddings_file, map_location="cpu", weights_only=False)
image_embeddings = saved["image_embeddings"].float()
caption_embeddings = saved["caption_embeddings"].float()
chunk_embeddings = saved["chunk_embeddings"].float()

print("Image records:", len(image_records))
print("Text chunks:", len(text_records))
print("Image vectors:", tuple(image_embeddings.shape))
print("Caption vectors:", tuple(caption_embeddings.shape))
print("Chunk vectors:", tuple(chunk_embeddings.shape))

Image records: 6
Text chunks: 30
Image vectors: (6, 512)
Caption vectors: (6, 512)
Chunk vectors: (30, 512)


## 5. Match image files

Match the uploaded image files to the image paths stored in the Day 2 metadata.

In [5]:
image_lookup = {Path(name).name: name for name in uploaded_images}
for record in image_records:
    filename = Path(record.get("image_path", "")).name
    record["uploaded_image_path"] = image_lookup.get(filename)
    record.setdefault("page", None)

print("Matched images:", sum(r["uploaded_image_path"] is not None for r in image_records),
      "of", len(image_records))

Matched images: 0 of 6


## 6. Check embedding dimensions

Confirm all three embedding sets use the same CLIP vector dimension.

In [6]:
dimensions = [
    image_embeddings.shape[1],
    caption_embeddings.shape[1],
    chunk_embeddings.shape[1]
]
print("Embedding dimensions:", dimensions)
if len(set(dimensions)) != 1:
    raise ValueError("Embedding dimensions do not match. Check the Day 2 outputs.")

Embedding dimensions: [512, 512, 512]


## 7. Create vector records

Combine text-chunk, caption, and image vectors. Store metadata for each vector, including source document, page when available, and image path.

In [7]:
vector_parts = [chunk_embeddings, caption_embeddings, image_embeddings]
vectors = torch.cat(vector_parts, dim=0).numpy().astype("float32")
metadata = []

for i, record in enumerate(text_records):
    metadata.append({
        "record_id": f"text_{i}", "modality": "text",
        "content_type": record.get("type", "text"),
        "source_doc": record.get("source_doc"), "page": record.get("page"),
        "text": record.get("text"), "image_path": None
    })

for i, record in enumerate(image_records):
    metadata.append({
        "record_id": f"caption_{i}", "modality": "caption",
        "content_type": "image_caption",
        "source_doc": record.get("source_doc"), "page": record.get("page"),
        "text": record.get("caption"), "image_path": record.get("uploaded_image_path")
    })

for i, record in enumerate(image_records):
    metadata.append({
        "record_id": f"image_{i}", "modality": "image",
        "content_type": "image",
        "source_doc": record.get("source_doc"), "page": record.get("page"),
        "text": record.get("caption"), "image_path": record.get("uploaded_image_path")
    })

if len(metadata) != len(vectors):
    raise ValueError("Vector count and metadata count do not match.")

print("Total vectors:", len(vectors))
print("Metadata records:", len(metadata))

Total vectors: 42
Metadata records: 42


## 8. Build the FAISS index

Normalize the vectors and create an inner-product index, which ranks cosine similarity for normalized vectors.

In [8]:
faiss.normalize_L2(vectors)
index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

print("Vectors indexed:", index.ntotal)
print("Embedding dimension:", index.d)

Vectors indexed: 42
Embedding dimension: 512


## 9. Load CLIP

Load the same CLIP checkpoint used in Day 2 so query embeddings are compatible with the index.

In [9]:
clip_model_id = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_model_id)
clip_model = CLIPModel.from_pretrained(clip_model_id).to(device)
clip_model.eval()
print("Loaded:", clip_model_id)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Loaded: openai/clip-vit-base-patch32


## 10. Define text query embeddings

Convert a search query into a normalized CLIP vector.

In [10]:
@torch.inference_mode()
def embed_query_text(query):
    inputs = clip_processor(
        text=[query], return_tensors="pt", padding=True, truncation=True
    ).to(device)
    outputs = clip_model.text_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )
    features = clip_model.text_projection(outputs.pooler_output)
    features = F.normalize(features, p=2, dim=-1)
    return features.cpu().numpy().astype("float32")

## 11. Search the multimodal index

Search text, caption, and image vectors with one text query and return matching provenance metadata.

In [11]:
def search_index(query, top_k=5):
    query_vector = embed_query_text(query)
    faiss.normalize_L2(query_vector)
    scores, indices = index.search(query_vector, min(top_k, index.ntotal))
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            result = metadata[int(idx)].copy()
            result["similarity"] = round(float(score), 4)
            results.append(result)
    return results

query = "What equipment was supplied to the customer?"
search_results = search_index(query, top_k=8)
display(pd.DataFrame(search_results))

,record_id,modality,content_type,source_doc,page,text,image_path,similarity
0,text_7,text,text,sample_multimodal_invoice.pdf,1,Items and Services,None,0.9032
1,text_2,text,text,sample_multimodal_invoice.pdf,1,Supplier,None,0.9009
2,text_11,text,text,sample_multimodal_invoice.pdf,1,Grand Total,None,0.8890
3,text_21,text,text,sample_multimodal_invoice.pdf,2,Service Summary,None,0.8839
4,text_13,text,text,sample_multimodal_invoice.pdf,1,Notes: Goods supplied are covered by the manuf...,None,0.8820
5,text_1,text,text,sample_multimodal_invoice.pdf,1,Original for recipient,None,0.8762
6,text_14,text,text,sample_multimodal_invoice.pdf,1,Authorised Signatory,None,0.8729
7,text_17,text,text,sample_multimodal_invoice.pdf,2,Service and Delivery Details,None,0.8707


## 12. Display retrieved images

Display the actual image files for image results when those files were uploaded.

In [16]:
image_results = [
    r for r in search_results
    if r["modality"] == "image" and r.get("image_path")
]
if image_results:
    for result in image_results:
        print(result["source_doc"], "| page:", result["page"],
              "| similarity:", result["similarity"])
        print(result["text"])
        display(Image.open(result["image_path"]).convert("RGB"))
else:
    print("No retrieved image file available. Upload extracted image files in Cell 3.")

No retrieved image file available. Upload extracted image files in Cell 3.


## 13. Review indexed metadata

Check how many text, caption, and image vectors were added and whether provenance fields are present.

In [13]:
metadata_df = pd.DataFrame(metadata)
display(metadata_df.groupby(["modality", "content_type"]).size().reset_index(name="count"))
display(metadata_df.head(10))
print("Records with source document:", metadata_df["source_doc"].notna().sum())
print("Text records with page:", metadata_df[metadata_df.modality == "text"]["page"].notna().sum())
print("Image/caption records with page:",
      metadata_df[metadata_df.modality.isin(["image", "caption"])]["page"].notna().sum())

,modality,content_type,count
0,caption,image_caption,6
1,image,image,6
2,text,table,5
3,text,text,25


,record_id,modality,content_type,source_doc,page,text,image_path
0,text_0,text,text,sample_multimodal_invoice.pdf,1.0,TAX INVOICE,None
1,text_1,text,text,sample_multimodal_invoice.pdf,1.0,Original for recipient,None
2,text_2,text,text,sample_multimodal_invoice.pdf,1.0,Supplier,None
3,text_3,text,text,sample_multimodal_invoice.pdf,1.0,"Meridian Research Labs Pvt. Ltd. Plot 18, Sect...",None
4,text_4,text,text,sample_multimodal_invoice.pdf,1.0,Bill To,None
5,text_5,text,text,sample_multimodal_invoice.pdf,1.0,"Northstar Office Supplies Unit 402, Tech Park ...",None
6,text_6,text,table,sample_multimodal_invoice.pdf,1.0,Invoice No. MRL-2026-1184 Invoice Date 18 Nov ...,None
7,text_7,text,text,sample_multimodal_invoice.pdf,1.0,Items and Services,None
8,text_8,text,table,sample_multimodal_invoice.pdf,1.0,# Description HSN/SAC Qty Unit Price (n) Amoun...,None
9,text_9,text,text,sample_multimodal_invoice.pdf,1.0,SGST (9%),None


Records with source document: 42
Text records with page: 30
Image/caption records with page: 0


## 14. Save the index and metadata

Save the FAISS index and its metadata sidecar for use in Day 4.

In [14]:
output_dir = Path("day3_index")
output_dir.mkdir(exist_ok=True)
faiss.write_index(index, str(output_dir / "multimodal_faiss.index"))
with open(output_dir / "multimodal_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("Saved FAISS index and metadata to", output_dir)

Saved FAISS index and metadata to day3_index


## 15. Download index files

Download both files together; the metadata file maps FAISS result positions to their source content.

In [15]:
files.download(str(output_dir / "multimodal_faiss.index"))
files.download(str(output_dir / "multimodal_metadata.json"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notebook summary

This notebook combines the Day 2 text-chunk, caption, and image embeddings in a single FAISS index. Each vector is linked to metadata describing its modality, source document, page when available, text or caption, and image path when the corresponding image file was uploaded. It tests text-query retrieval across all indexed modalities and saves the FAISS index with its metadata sidecar for Day 4.